In [1]:
import MDAnalysis as mda
import matplotlib.pyplot as plt
import numpy as np
import os
import re
from glob import glob 
import random
from braceexpand import braceexpand

ModuleNotFoundError: No module named 'braceexpand'

In [2]:
files= [("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.xtc"),
    ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.xtc"),
        ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.xtc"),
       ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep4.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep4.xtc"),
       ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep5.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep5.xtc"),
       ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep6.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep6.xtc"),
       ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep7.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep7.xtc"),
       ("/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep8.tpr"
         ,"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep8.xtc")]

In [3]:
def build_com_matrix(files, sel="resname BDN", ref="resname POPE"):
    all_com = []

    for top, traj in files:
        u = mda.Universe(top, traj)
        com = get_com(u, sel, ref)
        all_com.append(com)

    # pad naar max lengte (robust!)
    max_len = max(c.shape[0] for c in all_com)

    arr = np.full((len(all_com), max_len, 3), np.nan)

    for i, com in enumerate(all_com):
        arr[i, :len(com), :] = com

    return arr

In [4]:
def get_com(u, sel: str, ref: str):
    n_frames = len(u.trajectory)
    com_arr = np.empty((n_frames, 3))

    sel_group = u.select_atoms(sel)
    ref_group = u.select_atoms(ref)

    for i, ts in enumerate(u.trajectory):
        com_sel = sel_group.center_of_mass(unwrap=True)
        com_ref = ref_group.center_of_mass()
        com_arr[i] = com_sel - com_ref

    return com_arr

In [5]:
BDN_com_all_reps = np.empty((len(files), 5001, 3))

for i, simulation in enumerate(files):
    topology, trajectory = simulation
    u = mda.Universe(topology, trajectory)
    print(u.trajectory)
    BDN_com_all_reps[i, :, :] = get_com(u, sel = "resname BDN", ref = "resname POPE")

<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.xtc with 5001 frames of 43973 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.xtc with 5001 frames of 43892 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.xtc with 5001 frames of 43874 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep4.xtc with 5001 frames of 43793 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep5.xtc with 5001 frames of 43862 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep6.xtc with 5001 frames of 43874 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep7.xtc with 5001 frames of 43895 atoms>
<XTCReader /scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep8.xtc with 5001 frames of 43844 atoms>


In [6]:
def get_close_idx_com(com_all, target_z, tol):
    z = com_all[:, :, 2]

    idx = np.argwhere(np.isclose(z, target_z, atol=tol))
    n = len(idx)

    if n == 0:
        return None, None, 0

    choice = idx[np.random.randint(n)]
    value = z[choice[0], choice[1]]

    return value, choice, n

In [7]:
def write_frame(topology, trajectory, frame_idx, output_file):
    u = mda.Universe(topology, trajectory)
    u.trajectory[frame_idx]

    with mda.Writer(output_file, u.atoms.n_atoms) as w:
        w.write(u.atoms)

    return output_file

In [8]:
def get_atom_indices(structure_file, selection):
    u = mda.Universe(structure_file)
    sel = u.select_atoms(selection)

    return (sel.indices + 1).tolist()

In [9]:
def to_plumed_ranges(atom_list):
    ranges = []
    start = prev = atom_list[0]

    for a in atom_list[1:]:
        if a == prev + 1:
            prev = a
        else:
            ranges.append(f"{start}-{prev}" if start != prev else f"{start}")
            start = prev = a

    ranges.append(f"{start}-{prev}" if start != prev else f"{start}")
    return ",".join(ranges)

In [10]:
def write_plumed(plumedfile, where, structure_file):
    name = os.path.basename(plumedfile).replace(".dat", "")

    bdq_atoms = get_atom_indices(structure_file, "resname BDN")
    all_atoms = get_atom_indices(structure_file, "resname POPE")

    bdq_sel = to_plumed_ranges(bdq_atoms)
    all_sel = to_plumed_ranges(all_atoms)

    with open(plumedfile, "w") as f:
        f.write("UNITS LENGTH=A\n")
        f.write(f"WHOLEMOLECULES ENTITY0={bdq_sel}\n")
        f.write(f"MOLINFO STRUCTURE={name}.pdb\n")

        f.write(f"pop: COM ATOMS={all_sel}\n")
        f.write(f"bdq: COM ATOMS={bdq_sel}\n")

        f.write("d1: DISTANCE ATOMS=pop,bdq COMPONENTS\n")

        f.write(f"RESTRAINT ARG=d1.z KAPPA=5.0 AT={where}\n")
        f.write(f"PRINT ARG=d1.z FILE={name}-colvar_5.dat STRIDE=1\n")

In [13]:
target_z_values = np.arange(-32, 32, 0.5)
tolerance = 0.1

mapping_file = "umbrella_snapshot_mapping_bdn.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations_2/frame_z{i+1}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed_2/plumed_{i+1}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+1},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")



z=-32.0 | rep=1 frame=137 matches=20
z=-31.5 | rep=7 frame=523 matches=31
z=-31.0 | rep=7 frame=472 matches=39
z=-30.5 | rep=7 frame=2199 matches=51
z=-30.0 | rep=7 frame=525 matches=85
z=-29.5 | rep=7 frame=700 matches=85
z=-29.0 | rep=0 frame=902 matches=83
z=-28.5 | rep=0 frame=795 matches=137
z=-28.0 | rep=0 frame=833 matches=115
z=-27.5 | rep=0 frame=1008 matches=135
z=-27.0 | rep=7 frame=181 matches=114
z=-26.5 | rep=7 frame=2177 matches=115
z=-26.0 | rep=0 frame=844 matches=83
z=-25.5 | rep=7 frame=1535 matches=78
z=-25.0 | rep=3 frame=176 matches=52
z=-24.5 | rep=5 frame=10 matches=47
z=-24.0 | rep=0 frame=636 matches=33
z=-23.5 | rep=7 frame=105 matches=35
z=-23.0 | rep=0 frame=522 matches=19
z=-22.5 | rep=3 frame=177 matches=14
z=-22.0 | rep=0 frame=1087 matches=15
z=-21.5 | rep=5 frame=207 matches=7
z=-21.0 | rep=5 frame=3562 matches=8
z=-20.5 | rep=1 frame=706 matches=6
z=-20.0 | rep=7 frame=2254 matches=11
z=-19.5 | rep=1 frame=978 matches=18
z=-19.0 | rep=5 frame=3503 mat

In [24]:
print(BDN_com_all_reps[:, :, 2].min(), BDN_com_all_reps[:, :, 2].max())

-38.392741352427485 42.67966635485301


In [11]:
target_z_values = np.arange(-38, -17, 1)
tolerance = 0.125

mapping_file = "umbrella_snapshot_mapping_bis.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+33}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+33}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+33},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=-38 | rep=7 frame=757 matches=1
z=-37 | rep=6 frame=325 matches=1
z=-36 | rep=7 frame=1997 matches=1
z=-35 | rep=7 frame=2018 matches=3
z=-34 | rep=0 frame=1035 matches=8
z=-33 | rep=7 frame=1230 matches=14
z=-32 | rep=7 frame=476 matches=24
z=-31 | rep=7 frame=1193 matches=51
z=-30 | rep=7 frame=1736 matches=103
z=-29 | rep=0 frame=1430 matches=113
z=-28 | rep=0 frame=903 matches=143
z=-27 | rep=7 frame=1540 matches=138
z=-26 | rep=0 frame=1454 matches=109
z=-25 | rep=7 frame=34 matches=63
z=-24 | rep=7 frame=379 matches=38
z=-23 | rep=1 frame=156 matches=27
z=-22 | rep=7 frame=1276 matches=16
z=-21 | rep=7 frame=1488 matches=9
z=-20 | rep=3 frame=1146 matches=16
z=-19 | rep=3 frame=203 matches=36
z=-18 | rep=7 frame=2268 matches=61


In [12]:
target_z_values = np.arange(17, 38, 1)
tolerance = 0.125

mapping_file = "umbrella_snapshot_mapping_bis.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+54}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+54}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+54},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=17 | rep=4 frame=3370 matches=108
z=18 | rep=0 frame=3009 matches=64
z=19 | rep=6 frame=954 matches=27
z=20 | rep=2 frame=1263 matches=25
z=21 | rep=2 frame=1266 matches=24
z=22 | rep=6 frame=920 matches=20
z=23 | rep=2 frame=777 matches=47
z=24 | rep=6 frame=263 matches=57
z=25 | rep=0 frame=2719 matches=103
z=26 | rep=0 frame=2361 matches=136
z=27 | rep=6 frame=125 matches=148
z=28 | rep=0 frame=1830 matches=169
z=29 | rep=7 frame=785 matches=151
z=30 | rep=0 frame=79 matches=113
z=31 | rep=2 frame=550 matches=68
z=32 | rep=6 frame=827 matches=30
z=33 | rep=0 frame=1480 matches=13
z=34 | rep=6 frame=741 matches=3
z=35 | rep=7 frame=975 matches=3
z=36 | rep=6 frame=5 matches=1
z=37 | rep=6 frame=315 matches=1


In [ ]:
BDN_com_all_reps = np.empty((len(files), 5001, 3))

for i, simulation in enumerate(files):
    topology, trajectory = simulation
    u = mda.Universe(topology, trajectory)
    print(u.trajectory)
    BDN_com_all_reps[i, :, :] = get_com(u, sel = "resname BDN", ref = "resname POPE")

In [ ]:
target_z_values = np.arange(-38, -17, 1)
tolerance = 0.125

mapping_file = "umbrella_snapshot_mapping_bis.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+33}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+33}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+36},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

In [15]:
target_z_values = np.arange(-36, -30.5, 0.5)
tolerance = 0.075

mapping_file = "umbrella_snapshot_mapping_half.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+74}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+74}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+74},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=-36.0 | rep=7 frame=1997 matches=1
z=-35.5 | rep=7 frame=1631 matches=3
z=-35.0 | rep=7 frame=2018 matches=2
z=-34.5 | rep=7 frame=1229 matches=1
z=-34.0 | rep=7 frame=2022 matches=6
z=-33.5 | rep=7 frame=1228 matches=10
z=-33.0 | rep=7 frame=1230 matches=7
z=-32.5 | rep=0 frame=425 matches=15
z=-32.0 | rep=7 frame=546 matches=15
z=-31.5 | rep=7 frame=538 matches=24
z=-31.0 | rep=0 frame=1345 matches=26


In [ ]:
target_z_values = np.arange(-36, -30.5, 0.5)
tolerance = 0.075

mapping_file = "umbrella_snapshot_mapping_bis.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+84}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+84}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+84},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

In [16]:
target_z_values = np.arange(-19, -15, 0.5)
tolerance = 0.075

mapping_file = "umbrella_snapshot_mapping_half.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+85}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+85}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+85},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=-19.0 | rep=1 frame=512 matches=22
z=-18.5 | rep=1 frame=1005 matches=24
z=-18.0 | rep=3 frame=212 matches=32
z=-17.5 | rep=7 frame=2339 matches=65
z=-17.0 | rep=5 frame=225 matches=61
z=-16.5 | rep=7 frame=2302 matches=90
z=-16.0 | rep=5 frame=4238 matches=115
z=-15.5 | rep=1 frame=944 matches=156


In [18]:
target_z_values = np.arange(-12, 12, 0.5)
tolerance = 0.075

mapping_file = "umbrella_snapshot_mapping_half.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+93}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+93}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+93},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=-12.0 | rep=5 frame=1678 matches=337
z=-11.5 | rep=7 frame=4252 matches=355
z=-11.0 | rep=3 frame=3266 matches=304
z=-10.5 | rep=1 frame=408 matches=277
z=-10.0 | rep=6 frame=4083 matches=240
z=-9.5 | rep=3 frame=2125 matches=236
z=-9.0 | rep=5 frame=4921 matches=180
z=-8.5 | rep=3 frame=3796 matches=148
z=-8.0 | rep=5 frame=1590 matches=109
z=-7.5 | rep=3 frame=812 matches=113
z=-7.0 | rep=7 frame=4688 matches=72
z=-6.5 | rep=5 frame=1550 matches=58
z=-6.0 | rep=3 frame=4305 matches=42
z=-5.5 | rep=7 frame=4798 matches=24
z=-5.0 | rep=3 frame=4951 matches=24
z=-4.5 | rep=1 frame=1704 matches=11
z=-4.0 | rep=3 frame=2102 matches=14
z=-3.5 | rep=3 frame=4938 matches=15
z=-3.0 | rep=3 frame=4688 matches=11
z=-2.5 | rep=3 frame=4967 matches=10
z=-2.0 | rep=7 frame=4771 matches=8
z=-1.5 | rep=7 frame=4833 matches=11
z=-1.0 | rep=3 frame=4924 matches=11
z=-0.5 | rep=7 frame=4766 matches=14
z=0.0 | rep=1 frame=1665 matches=9
z=0.5 | rep=7 frame=4873 matches=14
z=1.0 | rep=7 frame=4880 matc

In [20]:
target_z_values = np.arange(12, 19, 0.5)
tolerance = 0.075

mapping_file = "umbrella_snapshot_mapping_half.csv"

# 1. COM matrix bouwen
BDN_com_all_reps = build_com_matrix(files)

# 2. loop over umbrella windows
for i, z_target in enumerate(target_z_values):

    value, idx, n = get_close_idx_com(
        BDN_com_all_reps,
        z_target,
        tolerance
    )

    if idx is None:
        print(f"Skipping {z_target} (0 matches)")
        continue

    rep, frame = idx
    topology, trajectory = files[rep]

    print(f"z={z_target} | rep={rep} frame={frame} matches={n}")

    # snapshot
    out_gro = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/configurations/frame_z{i+141}.gro"
    structure = write_frame(topology, trajectory, frame, out_gro)

    # plumed
    out_plumed = f"/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/umbrella_unbiased/plumed/plumed_{i+141}.dat"
    write_plumed(out_plumed, value, structure)

    # mapping
    with open(mapping_file, "a") as f:
        f.write(f"{i+141},{z_target},{rep + 1},{frame},{value},topol{rep + 1}.top,index{rep + 1}.ndx\n")

z=12.0 | rep=1 frame=3644 matches=348
z=12.5 | rep=0 frame=4343 matches=338
z=13.0 | rep=4 frame=4007 matches=318
z=13.5 | rep=1 frame=2625 matches=306
z=14.0 | rep=4 frame=231 matches=301
z=14.5 | rep=4 frame=952 matches=251
z=15.0 | rep=4 frame=4374 matches=183
z=15.5 | rep=4 frame=4795 matches=145
z=16.0 | rep=4 frame=381 matches=124
z=16.5 | rep=1 frame=3462 matches=85
z=17.0 | rep=6 frame=1762 matches=66
z=17.5 | rep=4 frame=96 matches=38
z=18.0 | rep=1 frame=4695 matches=46
z=18.5 | rep=4 frame=4339 matches=19
